<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

#### Handle class imbalance

In [4]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [5]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [6]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [7]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [9]:
# Initialize MLflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
experiment_name = "Anomaly Detection"
client = MlflowClient()
exp = client.get_experiment_by_name(experiment_name)
if exp is not None and exp.lifecycle_stage == "deleted":
    client.restore_experiment(exp.experiment_id)

mlflow.set_experiment(experiment_name)

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, name="model")
        else:
            mlflow.sklearn.log_model(model, name="model")  

2026/04/20 04:34:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/20 04:34:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/5ed0cebc2d544dc6b4d74cb0b7c6ba4a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/55f0b3b183954049a8eaa631a832e507
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/2/runs/450dab928f6e47c8bb27d63171d5e918
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/2/runs/9cfaa7370c9c4f12a98ea22588cffaaa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Register the Model

In [14]:
model_name = "XGB-Smote"
client = mlflow.MlflowClient()
exp = client.get_experiment_by_name("Anomaly Detection")

latest_run = client.search_runs(
    [exp.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=1,
)[0]

run_id = latest_run.info.run_id
model_uri = f"runs:/{run_id}/model"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
model_version = registered_model.version

print(f"Registered {model_name} version {model_version} from run {run_id}")

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/04/20 04:35:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 9cfaa7370c9c4f12a98ea22588cffaaa has no artifacts at artifact path 'model', registering model based on models:/m-e4b7fa93872f476981098971858124bc instead
2026/04/20 04:35:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 2


Registered XGB-Smote version 2 from run 9cfaa7370c9c4f12a98ea22588cffaaa


Created version '2' of model 'XGB-Smote'.


### Load the Model

In [15]:
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

### Transition the Model to Production

In [16]:
src_model_uri = f"models:/{model_name}/{model_version}"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
copied_model = client.copy_model_version(
    src_model_uri=src_model_uri,
    dst_name=production_model_name,
)
client.set_registered_model_alias(
    name=production_model_name,
    alias="champion",
    version=copied_model.version,
)

print(
    f"Copied {src_model_uri} to {production_model_name} version {copied_model.version} and set alias 'champion'"
    )

Registered model 'anomaly-detection-prod' already exists. Creating a new version of this model...
Copied version '2' of model 'XGB-Smote' to version '2' of model 'anomaly-detection-prod'.


Copied models:/XGB-Smote/2 to anomaly-detection-prod version 2 and set alias 'champion'


In [17]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

Please refer to following to learn more about model registry

https://mlflow.org/docs/latest/model-registry.html#model-registry-workflows to learn 